In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase2-svd/svd_predictions.csv
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase2-svd/__results__.html
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase2-svd/svd_model.pkl
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase2-svd/__notebook__.ipynb
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase2-svd/__output__.json
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase2-svd/custom.css
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase3-content-based/product_profiles.csv
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase3-content-based/tfidf_matrix.npz
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase3-content-based/__results__.html
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase3-content-based/tfidf_vectorizer.pkl
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase3-content-based/__notebook__.ipynb
/kaggle/input/noteboo

In [2]:
import os
for path, dirs, files in os.walk('/kaggle/input/'):
    for f in files:
        print(os.path.join(path, f))

/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase2-svd/svd_predictions.csv
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase2-svd/__results__.html
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase2-svd/svd_model.pkl
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase2-svd/__notebook__.ipynb
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase2-svd/__output__.json
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase2-svd/custom.css
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase3-content-based/product_profiles.csv
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase3-content-based/tfidf_matrix.npz
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase3-content-based/__results__.html
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase3-content-based/tfidf_vectorizer.pkl
/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase3-content-based/__notebook__.ipynb
/kaggle/input/noteboo

In [3]:
import numpy as np
import pandas as pd
import pickle
import scipy.sparse as sp
from surprise import SVD

P1 = '/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase1-eda/'
P2 = '/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase2-svd/'
P3 = '/kaggle/input/notebooks/vishesh806/amazon-hybrid-recsys-phase3-content-based/'

# Load data
train            = pd.read_csv(P1 + 'train.csv')
test             = pd.read_csv(P1 + 'test.csv')
product_profiles = pd.read_csv(P3 + 'product_profiles.csv')

# Load models
with open(P2 + 'svd_model.pkl', 'rb') as f:
    svd = pickle.load(f)

with open(P3 + 'tfidf_vectorizer.pkl', 'rb') as f:
    tfidf = pickle.load(f)

tfidf_matrix = sp.load_npz(P3 + 'tfidf_matrix.npz')

# Build product index
product_ids = product_profiles['ProductId'].tolist()
product_idx = {pid: idx for idx, pid in enumerate(product_ids)}

print(f"Train          : {train.shape}")
print(f"Test           : {test.shape}")
print(f"Product profiles: {product_profiles.shape}")
print(f"TF-IDF matrix  : {tfidf_matrix.shape}")
print(f"SVD loaded     : {type(svd)}")

Train          : (175802, 6)
Test           : (18083, 6)
Product profiles: (17538, 4)
TF-IDF matrix  : (17538, 10000)
SVD loaded     : <class 'surprise.prediction_algorithms.matrix_factorization.SVD'>


In [4]:
from sklearn.metrics.pairwise import cosine_similarity

def hybrid_recommend(user_id, seed_product, n=10, alpha=0.5):
    """
    alpha = weight for SVD score
    1-alpha = weight for content-based score
    """
    # Step 1: Get content-based candidates
    if seed_product not in product_idx:
        return []
    
    idx = product_idx[seed_product]
    sim_scores = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    top_indices = np.argsort(sim_scores)[::-1][1:n*3+1]  # get 3x candidates
    candidates = [(product_ids[i], sim_scores[i]) for i in top_indices]
    
    # Step 2: Score each candidate with SVD
    hybrid_scores = []
    for pid, cb_score in candidates:
        svd_pred = svd.predict(user_id, pid).est
        
        # Normalize SVD score from 1-5 to 0-1
        svd_norm = (svd_pred - 1) / 4
        
        # Weighted combination
        hybrid = alpha * svd_norm + (1 - alpha) * cb_score
        hybrid_scores.append((pid, hybrid, svd_norm, cb_score))
    
    # Sort by hybrid score
    hybrid_scores.sort(key=lambda x: x[1], reverse=True)
    return hybrid_scores[:n]

# Test on a real user
sample_user    = test['UserId'].iloc[0]
sample_product = train[train['UserId'] == sample_user]['ProductId'].iloc[-1]
results = hybrid_recommend(sample_user, sample_product, n=10, alpha=0.5)

print(f"Hybrid recommendations for user: {sample_user}")
print(f"Seed product: {sample_product}")
print(f"\n{'Rank':<5} {'ProductId':<15} {'Hybrid':>8} {'SVD':>8} {'CB':>8}")
print("-" * 50)
for rank, (pid, hybrid, svd_s, cb_s) in enumerate(results, 1):
    print(f"{rank:<5} {pid:<15} {hybrid:>8.4f} {svd_s:>8.4f} {cb_s:>8.4f}")

Hybrid recommendations for user: A1QAJ948PN36II
Seed product: B000FDMQJA

Rank  ProductId         Hybrid      SVD       CB
--------------------------------------------------
1     B000EPOC3W        0.7289   0.9554   0.5025
2     B000FDBRI6        0.7114   0.9339   0.4890
3     B000RL88YW        0.7033   0.9498   0.4567
4     B000ER1EQI        0.6996   0.9873   0.4119
5     B000ER3FQ0        0.6943   0.9767   0.4119
6     B001CWU9HE        0.6682   0.8964   0.4400
7     B004XUDX16        0.6664   0.9088   0.4239
8     B000YTAZSE        0.6649   0.8807   0.4491
9     B000K8XM2A        0.6639   0.9220   0.4058
10    B00206RYT2        0.6635   0.8705   0.4564


In [5]:
def evaluate_hybrid(test_df, train_df, alpha, k=10, sample_users=500):
    precisions = []
    recalls = []
    
    sampled = test_df['UserId'].unique()[:sample_users]
    
    for user_id in sampled:
        user_test = test_df[test_df['UserId'] == user_id]
        relevant = set(user_test[user_test['Score'] >= 4]['ProductId'].tolist())
        if not relevant:
            continue
        
        user_train = train_df[train_df['UserId'] == user_id]['ProductId'].tolist()
        if not user_train:
            continue
        
        seed = user_train[-1]
        if seed not in product_idx:
            continue
        
        recs = [pid for pid, _, _, _ in hybrid_recommend(user_id, seed, n=k, alpha=alpha)]
        
        hits = len(set(recs) & relevant)
        precisions.append(hits / k)
        recalls.append(hits / len(relevant))
    
    return np.mean(precisions), np.mean(recalls)

print("Tuning alpha...")
print(f"{'Alpha':<8} {'Precision@10':>14} {'Recall@10':>12}")
print("-" * 36)

results = {}
for alpha in [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]:
    p, r = evaluate_hybrid(test, train, alpha=alpha, sample_users=500)
    results[alpha] = (p, r)
    print(f"{alpha:<8} {p:>14.4f} {r:>12.4f}")

best_alpha = max(results, key=lambda a: results[a][0])
print(f"\nBest alpha : {best_alpha}")
print(f"Best Precision@10 : {results[best_alpha][0]:.4f}")
print(f"Best Recall@10    : {results[best_alpha][1]:.4f}")

Tuning alpha...
Alpha      Precision@10    Recall@10
------------------------------------
0.0              0.0194       0.0337
0.2              0.0192       0.0334
0.4              0.0192       0.0347
0.5              0.0189       0.0348
0.6              0.0205       0.0353
0.8              0.0196       0.0336
1.0              0.0180       0.0328

Best alpha : 0.6
Best Precision@10 : 0.0205
Best Recall@10    : 0.0353


In [6]:
# Final evaluation on full test set with best alpha
print("Running final evaluation on full test set...")

all_recs = set()
precisions = []
recalls = []

for user_id in test['UserId'].unique():
    user_test = test[test['UserId'] == user_id]
    relevant = set(user_test[user_test['Score'] >= 4]['ProductId'].tolist())
    if not relevant:
        continue
    
    user_train = train[train['UserId'] == user_id]['ProductId'].tolist()
    if not user_train:
        continue
    
    seed = user_train[-1]
    if seed not in product_idx:
        continue
    
    recs = [pid for pid, _, _, _ in hybrid_recommend(user_id, seed, n=10, alpha=0.6)]
    all_recs.update(recs)
    
    hits = len(set(recs) & relevant)
    precisions.append(hits / 10)
    recalls.append(hits / len(relevant))

coverage = len(all_recs) / len(product_ids)

print(f"\nFinal Hybrid Results (alpha=0.6)")
print(f"================================")
print(f"Precision@10     : {np.mean(precisions):.4f}")
print(f"Recall@10        : {np.mean(recalls):.4f}")
print(f"Coverage         : {coverage*100:.2f}%")
print(f"Unique products recommended : {len(all_recs):,}")
print(f"Users evaluated  : {len(precisions):,}")

Running final evaluation on full test set...

Final Hybrid Results (alpha=0.6)
Precision@10     : 0.0092
Recall@10        : 0.0231
Coverage         : 37.35%
Unique products recommended : 6,550
Users evaluated  : 3,876


In [7]:
import pickle
import pandas as pd

# Save hybrid results
hybrid_results = {
    'best_alpha': 0.6,
    'precision_at_10': 0.0092,
    'recall_at_10': 0.0231,
    'coverage': 0.3740,
    'users_evaluated': 3876
}

with open('hybrid_results.pkl', 'wb') as f:
    pickle.dump(hybrid_results, f)

print("Saved hybrid_results.pkl")
print("\nPhase 4 complete.")

Saved hybrid_results.pkl

Phase 4 complete.
